In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
import shap
from typing import Dict, Any, List, Optional

warnings.filterwarnings('ignore')

class AdvancedRiskSimulatorModel:
    """
    Core ML Engine for the Smart Student Analytics Platform.
    Extended with SHAP explanations, dynamic interventions, feature alignment, and delta tracking.
    """
    
    def __init__(self, data_path: str, rf_path: str, kmeans_path: str, le_path: str):
        # 1. Load Assets
        self.df = pd.read_csv(data_path)
        self.rf_model = joblib.load(rf_path)
        self.kmeans_model = joblib.load(kmeans_path)
        self.label_encoder = joblib.load(le_path)
        
        # 2. Extract Feature Footprints (Validates inputs dynamically)
        self.rf_features = list(self.rf_model.feature_names_in_)
        # Fallback for KMeans if feature_names_in_ is missing in older sklearn versions
        self.kmeans_features = getattr(self.kmeans_model, 'feature_names_in_', self.rf_features)
        
        # 3. Initialize SHAP Explainer
        # TreeExplainer is highly optimized for Random Forest
        self.explainer = shap.TreeExplainer(self.rf_model)
        
        # 4. Define State
        self.student_id = None
        self.baseline_state = None
        self.simulated_state = None
        self.peer_baseline = None
        
        self.pass_idx = list(self.label_encoder.classes_).index('Pass')

    def load_student(self, student_id: int) -> bool:
        """Loads the baseline state for a specific student and calculates peer metrics."""
        student_data = self.df[self.df['id_student'] == student_id]
        if student_data.empty:
            raise ValueError(f"Student ID {student_id} not found in dataset.")
            
        self.student_id = student_id
        self.baseline_state = student_data.iloc[0].copy()
        self.simulated_state = self.baseline_state.copy()
        
        self._calculate_peer_baseline()
        return True

    def _align_features(self, state: pd.Series, required_features: List[str]) -> pd.DataFrame:
        """
        Critical Validation: Ensures the model receives exactly the columns it expects,
        in the exact order, filling missing downstream features with 0.
        """
        return pd.DataFrame([state]).reindex(columns=required_features, fill_value=0)

    def _calculate_peer_baseline(self):
        """Calculates the median stats of students in the same module who Passed."""
        module = self.baseline_state['code_module']
        successful_peers = self.df[
            (self.df['code_module'] == module) & 
            (self.df['final_result'].isin(['Pass', 'Distinction']))
        ]
        
        metrics = ['active_days', 'total_clicks', 'assessment_mean_score', 'early_engagement_ratio']
        self.peer_baseline = successful_peers[metrics].median().to_dict()

    def apply_simulation(self, modifications: Dict[str, float]):
        """Applies direct numerical adjustments to the simulated state."""
        for feature, new_value in modifications.items():
            if feature in self.simulated_state.index:
                self.simulated_state[feature] = new_value

    def run_sensitivity_analysis(self, perturbation_pct: float = 0.10) -> List[Dict[str, Any]]:
        """
        Perturbation Analysis: Bumps key features by a configurable percentage.
        Maintains internal consistency for dependent engineered features.
        """
        base_df = self._align_features(self.baseline_state, self.rf_features)
        base_pass_prob = self.rf_model.predict_proba(base_df)[0][self.pass_idx]

        features_to_test = ['active_days', 'assessment_mean_score', 'total_clicks', 'clicks_early']
        impact_scores = []

        for feat in features_to_test:
            test_state = self.baseline_state.copy()
            
            # Apply configurable perturbation
            increase_amount = test_state[feat] * perturbation_pct
            test_state[feat] += increase_amount
            
            # Feature Consistency Engine
            if feat == 'assessment_mean_score':
                test_state[feat] = min(100.0, test_state[feat])
            elif feat == 'clicks_early':
                # If early clicks increase, total clicks must increase
                test_state['total_clicks'] += increase_amount
                if test_state['total_clicks'] > 0:
                    test_state['early_engagement_ratio'] = test_state['clicks_early'] / test_state['total_clicks']
            elif feat == 'total_clicks':
                # Rebalance ratio if total clicks artificially pumped
                if test_state['total_clicks'] > 0:
                    test_state['early_engagement_ratio'] = test_state['clicks_early'] / test_state['total_clicks']
                
            test_df = self._align_features(test_state, self.rf_features)
            new_prob = self.rf_model.predict_proba(test_df)[0][self.pass_idx]
            
            impact_scores.append({
                'feature': feat,
                'lift': float(new_prob - base_pass_prob),
                'simulated_value': float(test_state[feat])
            })
            
        return sorted(impact_scores, key=lambda x: x['lift'], reverse=True)

    def _compute_shap(self, state: pd.Series, predicted_class_idx: int) -> Dict[str, Any]:
        """Generates Local SHAP explanations for the specific prediction."""
        df_aligned = self._align_features(state, self.rf_features)
        shap_values = self.explainer.shap_values(df_aligned)
        
        # Handle different SHAP versions (list of arrays vs array)
        if isinstance(shap_values, list):
            class_shap = shap_values[predicted_class_idx][0]
        else:
            class_shap = shap_values[0, :, predicted_class_idx]
            
        # Create a mapping of feature to SHAP value
        feature_impacts = {feat: float(val) for feat, val in zip(self.rf_features, class_shap)}
        
        # Sort to find top positive and negative drivers
        sorted_impacts = sorted(feature_impacts.items(), key=lambda x: x[1], reverse=True)
        
        return {
            "top_positive_drivers": [{"feature": k, "impact": v} for k, v in sorted_impacts[:3]],
            "top_negative_drivers": [{"feature": k, "impact": v} for k, v in sorted_impacts[-3:]]
        }

    def _recompute_intervention(self, pred_label: str, cluster: int, behavior_index: float) -> Dict[str, str]:
        """Dynamic mapping of interventions based on the new simulated state."""
        # This replaces static recommendations with dynamic generation
        risk_level = "Low"
        priority = "Standard"
        recommendations = ["Maintain Learning Routine"]

        if pred_label in ['Fail', 'Withdrawn']:
            risk_level = "Critical"
            priority = "Immediate"
            recommendations = ["Schedule Academic Advising", "Enroll in Peer Tutoring", "Review Assessment Feedback"]
        elif pred_label == 'Pass' and behavior_index > 0.6:
            risk_level = "Medium"
            priority = "Monitor"
            recommendations = ["Skill Development Workshops", "Time Management Coaching"]

        return {
            "updated_risk_level": risk_level,
            "priority": priority,
            "recommendations": recommendations,
            "target_cluster": int(cluster)
        }

    def get_comprehensive_report(self) -> Dict[str, Any]:
        """
        Assembles all baseline, simulated, delta, SHAP, and intervention data 
        into a single, clean dictionary ready for JSON serialization.
        """
        # 1. Feature Alignment
        df_base_rf = self._align_features(self.baseline_state, self.rf_features)
        df_sim_rf = self._align_features(self.simulated_state, self.rf_features)
        df_base_km = self._align_features(self.baseline_state, list(self.kmeans_features))
        df_sim_km = self._align_features(self.simulated_state, list(self.kmeans_features))
        
        # 2. Base Predictions & Probabilities
        base_pred_idx = self.rf_model.predict(df_base_rf)[0]
        sim_pred_idx = self.rf_model.predict(df_sim_rf)[0]
        
        base_pred_lbl = self.label_encoder.inverse_transform([base_pred_idx])[0]
        sim_pred_lbl = self.label_encoder.inverse_transform([sim_pred_idx])[0]
        
        base_probs = self.rf_model.predict_proba(df_base_rf)[0]
        sim_probs = self.rf_model.predict_proba(df_sim_rf)[0]
        
        # 3. Probability Deltas
        prob_deltas = {}
        for i, cls in enumerate(self.label_encoder.classes_):
            prob_deltas[str(cls)] = {
                "baseline": float(base_probs[i]),
                "simulated": float(sim_probs[i]),
                "delta": float(sim_probs[i] - base_probs[i])
            }
            
        # 4. Clusters & Interventions
        base_cluster = int(self.kmeans_model.predict(df_base_km)[0])
        sim_cluster = int(self.kmeans_model.predict(df_sim_km)[0])
        
        updated_interventions = self._recompute_intervention(
            sim_pred_lbl, sim_cluster, float(self.simulated_state.get('behavior_risk_index', 0.5))
        )

        # 5. SHAP Explanations (Explaining the specific prediction made)
        base_shap = self._compute_shap(self.baseline_state, base_pred_idx)
        sim_shap = self._compute_shap(self.simulated_state, sim_pred_idx)

        # 6. Assemble JSON-Ready Dictionary
        return {
            "metadata": {
                "student_id": int(self.student_id),
                "module": str(self.baseline_state['code_module'])
            },
            "predictions": {
                "baseline_outcome": str(base_pred_lbl),
                "simulated_outcome": str(sim_pred_lbl),
                "outcome_changed": base_pred_lbl != sim_pred_lbl
            },
            "confidence_and_deltas": prob_deltas,
            "clustering": {
                "baseline_cluster": base_cluster,
                "simulated_cluster": sim_cluster,
                "cluster_transitioned": base_cluster != sim_cluster
            },
            "explainability_shap": {
                "baseline_drivers": base_shap,
                "simulated_drivers": sim_shap
            },
            "prescriptive_interventions": updated_interventions,
            "peer_benchmarks": self.peer_baseline,
            "sensitivity_analysis_15pct": self.run_sensitivity_analysis(perturbation_pct=0.15)
        }

c:\Users\vishw\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os

folder_path = 'C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish'
print("Files in this folder:")
for file in os.listdir(folder_path):
    print(f" - {file}")

Files in this folder:
 - .git
 - .gitattributes
 - .gitignore
 - .venv
 - assessments.csv
 - clustering.ipynb
 - clustering1.ipynb
 - courses.csv
 - data
 - dataset.txt
 - eda.ipynb
 - eda1.ipynb
 - eda_figures
 - feature_summary.csv
 - ml_pipeline.ipynb
 - models
 - oulad_master_features.csv
 - risk_simulator.ipynb
 - studentAssessment.csv
 - studentInfo.csv
 - studentRegistration.csv
 - studentVle.csv
 - vle.csv


In [5]:
import json

# 1. Initialize Engine using the sub-folders (data/ and models/)
engine = AdvancedRiskSimulatorModel(
    data_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/data/processed/student_clustered.csv',
    rf_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/final_random_forest.pkl',
    kmeans_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/final_kmeans.pkl',
    le_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/label_encoder.pkl'
)

# 2. Load Student
engine.load_student(student_id=11391) 

# 3. Apply a Simulation
engine.apply_simulation({
    'assessment_mean_score': 85.0,
    'active_days': engine.baseline_state['active_days'] + 15
})

# 4. Generate the Comprehensive JSON-ready Payload
api_response_payload = engine.get_comprehensive_report()

# 5. Verify Serialization
print(json.dumps(api_response_payload, indent=4))

{
    "metadata": {
        "student_id": 11391,
        "module": "AAA"
    },
    "predictions": {
        "baseline_outcome": "Pass",
        "simulated_outcome": "Pass",
        "outcome_changed": false
    },
    "confidence_and_deltas": {
        "Distinction": {
            "baseline": 0.13172770542044257,
            "simulated": 0.21050512231098417,
            "delta": 0.0787774168905416
        },
        "Fail": {
            "baseline": 0.01444208735342273,
            "simulated": 0.009151888374627173,
            "delta": -0.005290198978795556
        },
        "Pass": {
            "baseline": 0.8465305000333969,
            "simulated": 0.7698480032280597,
            "delta": -0.07668249680533723
        },
        "Withdrawn": {
            "baseline": 0.007299707192737933,
            "simulated": 0.0104949860863292,
            "delta": 0.003195278893591267
        }
    },
    "clustering": {
        "baseline_cluster": 3,
        "simulated_cluster": 3,
        

In [6]:
# ==========================================
# EXTENSION 1: ALGORITHMIC RECOURSE ENGINE
# "Minimum Viable Effort" Optimizer
# ==========================================
import copy

class PrescriptiveRiskSimulator(AdvancedRiskSimulatorModel):
    """
    Extends the base AdvancedRiskSimulatorModel with a Cost-Weighted 
    Greedy Optimization Algorithm to calculate the exact minimum 
    behavioral changes required to rescue a failing student.
    """
    
    def __init__(self, data_path: str, rf_path: str, kmeans_path: str, le_path: str):
        super().__init__(data_path, rf_path, kmeans_path, le_path)
        
        # Define the actionable space, realistic maximums, and effort costs
        # Cost Weight: Higher = harder for a student to achieve in real life
        self.action_space = {
            'assessment_mean_score': {'step': 2.0, 'max': 100.0, 'cost': 2.5}, 
            'active_days': {'step': 3.0, 'max': 150.0, 'cost': 1.0},
            'total_clicks': {'step': 30.0, 'max': 3000.0, 'cost': 0.5},
            'early_engagement_ratio': {'step': 0.05, 'max': 1.0, 'cost': 1.2}
        }

    def find_optimal_path_to_pass(self, max_iterations: int = 50) -> Dict[str, Any]:
        """
        Calculates the mathematical minimum viable effort to achieve a 'Pass'.
        """
        # 1. Check if optimization is even necessary
        base_df = self._align_features(self.baseline_state, self.rf_features)
        base_pred_idx = self.rf_model.predict(base_df)[0]
        base_pred_lbl = self.label_encoder.inverse_transform([base_pred_idx])[0]
        base_pass_prob = self.rf_model.predict_proba(base_df)[0][self.pass_idx]
        
        if base_pred_lbl in ['Pass', 'Distinction']:
            return {
                "status": "Optimization Not Required",
                "message": f"Student is already predicted to {base_pred_lbl}.",
                "current_probability": float(base_pass_prob)
            }

        # 2. Initialize the Optimization State
        current_state = self.baseline_state.copy()
        accumulated_deltas = {feat: 0.0 for feat in self.action_space.keys()}
        iteration_log = []
        
        # 3. Cost-Weighted Greedy Optimization Loop
        for iteration in range(max_iterations):
            best_move = None
            max_efficiency = 0.0  # Probability lift per unit of effort cost
            best_prob = 0.0
            
            # Test bumping each feature by its step size
            for feat, params in self.action_space.items():
                if current_state[feat] >= params['max']:
                    continue # Feature is already maxed out
                    
                test_state = current_state.copy()
                test_state[feat] = min(params['max'], test_state[feat] + params['step'])
                
                # Maintain feature integrity (if clicks go up, ratio adjusts slightly)
                if feat == 'total_clicks' and test_state['total_clicks'] > 0:
                    test_state['early_engagement_ratio'] = test_state['clicks_early'] / test_state['total_clicks']
                
                # Predict new probability
                test_df = self._align_features(test_state, self.rf_features)
                new_prob = self.rf_model.predict_proba(test_df)[0][self.pass_idx]
                
                # Calculate Efficiency Gradient (Delta Probability / Cost)
                prob_lift = new_prob - base_pass_prob
                efficiency = prob_lift / params['cost']
                
                if efficiency > max_efficiency:
                    max_efficiency = efficiency
                    best_move = feat
                    best_prob = new_prob
            
            # If no move improves the probability, the model is stuck (deeply failing)
            if best_move is None or max_efficiency <= 0:
                break
                
            # Apply the best move to the current state
            step_size = self.action_space[best_move]['step']
            current_state[best_move] = min(self.action_space[best_move]['max'], current_state[best_move] + step_size)
            accumulated_deltas[best_move] += step_size
            base_pass_prob = best_prob
            
            iteration_log.append({
                "iteration": iteration + 1,
                "action_taken": f"Increased {best_move} by {step_size}",
                "new_pass_probability": float(best_prob)
            })
            
            # Check if threshold is crossed (Prediction flips to Pass)
            current_df = self._align_features(current_state, self.rf_features)
            current_pred_lbl = self.label_encoder.inverse_transform(self.rf_model.predict(current_df))[0]
            
            if current_pred_lbl in ['Pass', 'Distinction']:
                # Success!
                self.simulated_state = current_state.copy() # Lock in the winning state
                
                return {
                    "status": "Optimization Successful",
                    "message": "Minimum behavioral changes calculated to achieve a Pass prediction.",
                    "final_prediction": current_pred_lbl,
                    "final_pass_probability": float(best_prob),
                    "required_behavioral_changes": {
                        feat: {
                            "original_value": float(self.baseline_state[feat]),
                            "target_value": float(current_state[feat]),
                            "required_increase": float(delta)
                        } for feat, delta in accumulated_deltas.items() if delta > 0
                    },
                    "optimization_path": iteration_log
                }

        # 4. Loop finished without reaching 'Pass'
        return {
            "status": "Deep Intervention Required",
            "message": "Even with maximum optimized effort, the model does not predict a Pass. Fundamental academic support needed.",
            "max_achieved_probability": float(base_pass_prob)
        }

In [8]:
# ==========================================
# EXECUTION: THE OPTIMIZER IN ACTION
# ==========================================
import json

# 1. Initialize the Upgraded Engine
optimizer_engine = PrescriptiveRiskSimulator(
    data_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/data/processed/student_clustered.csv',
    rf_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/final_random_forest.pkl',
    kmeans_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/final_kmeans.pkl',
    le_path='C:/Users/vishw/OneDrive/New folder/ITSP2026_Dhruv_Vish/models/label_encoder.pkl'         # Update absolute path if needed
)

# 2. Load a failing student (e.g., student ID 2679821 from your snippet)
FAILING_STUDENT_ID = 2679821 
optimizer_engine.load_student(student_id=FAILING_STUDENT_ID)

# 3. Execute Algorithmic Recourse
print("🔍 Running Cost-Weighted Minimal Effort Optimizer...")
optimization_report = optimizer_engine.find_optimal_path_to_pass()

# 4. Serialize output for the backend team
print("\n" + "="*60)
print(f"🎯 ALGORITHMIC RECOURSE REPORT FOR STUDENT {FAILING_STUDENT_ID}")
print("="*60)
print(json.dumps(optimization_report, indent=4))

🔍 Running Cost-Weighted Minimal Effort Optimizer...

🎯 ALGORITHMIC RECOURSE REPORT FOR STUDENT 2679821
{
    "status": "Deep Intervention Required",
    "message": "Even with maximum optimized effort, the model does not predict a Pass. Fundamental academic support needed.",
    "max_achieved_probability": 0.001190779119749805
}
